# review_report

> One self-contained HTML page for checking the visual result of a ticket or an evaluation run: images, overlays and metrics, without starting a kernel.

After a ticket the agent calls **one function** and gets **one `report.html`**. The reviewer opens it, switches layers on and off, and later (ticket #51) clicks on a problem to leave a comment. This notebook builds the core: the data unit (`Sample`), the section plug-in interface, and the first section, **overlay** (ground truth vs prediction vs errors).

This is the *only* HTML renderer in the library. Evaluation (#28) and embeddings (#39) will add sections; they will not build their own pages. Design: `docs/superpowers/specs/2026-10-08-review-report-design.md`.

**Constraints that shape every step**

- One file, no network at view time: CSS and JS are inline, images are base64.
- Masks are drawn as **SVG contours**, never JPEG, so they stay exact.
- Coordinates are in **original image pixels**; the page scales for display (ticket #51 depends on this).
- 16-bit images keep their raw values in a linked full-resolution file.
- Metrics come from the existing `SegMetrics`; nothing is re-implemented.

**Plan**

1. `Sample`: one image + ground truth + prediction, binary / multiclass / multilabel.
2. Dummy data: a seeded generator so everything below can be tested without real data.
3. Metrics: per-class Dice/IoU per sample and pooled, through `SegMetrics`.
4. Display: percentile window for 16-bit, JPEG thumbnail as base64.
5. Contours: mask to SVG path, checked by drawing the path back.
6. Sampling: worst / random / best groups with a fixed seed.
7. Sections: `SectionOutput`, `ReportContext`, registry.
8. Overlay section: layer toggles, per-class table, full-resolution link.
9. `build_report`: sections to one page plus `report.json`.
10. Adapter: model + dataset to `Sample`s (the real-data path).

In [ ]:
#| default_exp review_report

In [1]:
#| hide
from nbdev.showdoc import *

In [2]:
#| export
import base64
import json
import re
from dataclasses import dataclass, field
from html import escape as esc
from pathlib import Path
from typing import Callable, Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import torch


In [3]:
#| exporti
from segmentation_tools_pt.inference import decode, predict
from segmentation_tools_pt.metrics import SegMetrics

## 1. The unit of work: `Sample`

A `Sample` holds *stored* predictions, not a model. The repo's ranking helpers (`rank_samples`, `save_error_poster`) need a live model and a dataset, so the renderer cannot call them; keeping predictions in the sample also lets the evaluation cache (#31) feed the same page later.

Mask layouts follow the rest of the library:

| task | `gt` / `pred` | `class_names` |
|---|---|---|
| binary | `(H, W)` with 0/1 | optional, one name |
| multiclass | `(H, W)` integer labels | label `i` is `class_names[i]` (index 0 is background) |
| multilabel | `(K, H, W)` with 0/1 | one name per channel |

`image` keeps its raw dtype (`uint8`, `uint16`, float) so 16-bit data is not squashed before someone asks for it.

In [4]:
#| export
@dataclass
class Sample:
    """One image with its ground truth and prediction, ready to render.

    Example:
        >>> s = Sample('a', np.zeros((4, 4), np.uint8), np.eye(4, dtype=np.uint8), np.eye(4, dtype=np.uint8))
        >>> s.sample_id
        'a'
    """
    sample_id: str
    image: np.ndarray                          # (H, W) or (H, W, C), raw dtype
    gt: np.ndarray                             # (H, W) labels or (K, H, W) 0/1
    pred: np.ndarray                           # same layout as gt
    class_names: Optional[List[str]] = None    # see the layout table above
    score: Optional[float] = None              # quality, higher is better; None = mean Dice
    prob: Optional[np.ndarray] = None          # reserved for calibration views
    meta: dict = field(default_factory=dict)   # source, lot, camera, ... for slicing

Every later step needs the same thing from a sample: one boolean mask **per class**, whatever the task. `_class_stacks` normalises the three layouts into `(K, H, W)` stacks and rejects mismatched shapes early, naming the sample.

*Experiment:* a multiclass label map becomes one-hot stacks by comparing against `arange(K)`.

In [5]:
lab = np.array([[0, 1, 1], [2, 2, 0]])
stack = lab[None] == np.arange(3)[:, None, None]
print(stack.shape, stack.sum(axis=(1, 2)))      # pixels per class
assert stack.shape == (3, 2, 3) and stack.sum(axis=(1, 2)).tolist() == [2, 2, 2]
assert (stack.sum(0) == 1).all()                # every pixel belongs to exactly one class

(3, 2, 3) [2 2 2]


In [6]:
#| export
@dataclass
class _Stacks:
    task: str
    names: List[str]
    gt: np.ndarray       # (K, H, W) bool
    pred: np.ndarray     # (K, H, W) bool

def _class_stacks(s: Sample) -> _Stacks:
    "Normalise binary / multiclass / multilabel masks to per-class boolean stacks."
    gt, pred = np.asarray(s.gt), np.asarray(s.pred)
    if gt.shape != pred.shape:
        raise ValueError(f'{s.sample_id}: gt {gt.shape} and pred {pred.shape} differ')
    if np.asarray(s.image).shape[:2] != gt.shape[-2:]:
        raise ValueError(f'{s.sample_id}: image {np.asarray(s.image).shape[:2]} does not match mask {gt.shape[-2:]}')
    names = list(s.class_names) if s.class_names else None
    if gt.ndim == 3:
        names = names or [f'class_{i}' for i in range(gt.shape[0])]
        if len(names) != gt.shape[0]:
            raise ValueError(f'{s.sample_id}: {gt.shape[0]} mask channels but {len(names)} class names')
        return _Stacks('multilabel', names, gt > 0, pred > 0)
    if gt.ndim != 2:
        raise ValueError(f'{s.sample_id}: masks must be (H, W) or (K, H, W), got {gt.shape}')
    top = int(max(gt.max(), pred.max()))
    if top <= 1 and (names is None or len(names) == 1):
        return _Stacks('binary', names or ['foreground'], (gt > 0)[None], (pred > 0)[None])
    k = max(len(names or []), top + 1)
    names = (names or []) + [f'class_{i}' for i in range(len(names or []), k)]
    ids = np.arange(k)[:, None, None]
    return _Stacks('multiclass', names, gt[None] == ids, pred[None] == ids)

In [7]:
# verify: all three layouts, and the early errors
_img = np.zeros((4, 5), np.uint8)
m2 = np.zeros((4, 5), np.uint8); m2[1:3, 1:4] = 1
st = _class_stacks(Sample('b', _img, m2, m2));                              assert (st.task, st.names, st.gt.shape) == ('binary', ['foreground'], (1, 4, 5))
lab = m2.copy(); lab[0, 0] = 2
st = _class_stacks(Sample('m', _img, lab, lab, ['bg', 'x', 'y']));          assert (st.task, st.gt.shape) == ('multiclass', (3, 4, 5))
st = _class_stacks(Sample('l', _img, np.stack([m2, m2]), np.stack([m2, m2]))); assert (st.task, st.names) == ('multilabel', ['class_0', 'class_1'])
for bad in (Sample('x', _img, m2, m2[:3]), Sample('y', _img[:3], m2, m2), Sample('z', _img, np.stack([m2, m2]), np.stack([m2, m2]), ['only_one'])):
    try: _class_stacks(bad)
    except ValueError as e: print('ok:', e)
    else: raise AssertionError('expected ValueError')

ok: x: gt (4, 5) and pred (3, 5) differ
ok: y: image (3, 5) does not match mask (4, 5)
ok: z: 2 mask channels but 1 class names


## 2. Dummy data

Tests and docs need samples without real data. `make_dummy_samples` draws ellipses ("blob") and a thin bar on a noisy gradient, then damages the prediction by an amount that is different for every sample (shift, erosion, a false-positive circle, sometimes a total miss). The damage order is shuffled by the seed, so worst samples are not always last, and the same seed always gives the same samples.

*Experiment:* one damaged mask, to see the knobs behave.

In [8]:
rng = np.random.default_rng(0)
gt = np.zeros((60, 80), np.uint8); cv2.ellipse(gt, (40, 30), (14, 9), 20, 0, 360, 1, -1)
m = cv2.warpAffine(gt, np.float32([[1, 0, 4], [0, 1, -3]]), (80, 60), flags=cv2.INTER_NEAREST)   # shift by (4, -3)
m = cv2.erode(m, np.ones((3, 3), np.uint8), iterations=2)
inter = int((gt & m).sum()); dice = 2 * inter / (gt.sum() + m.sum())
print(f'gt={gt.sum()} px  damaged={m.sum()} px  dice={dice:.3f}')
assert 0.3 < dice < 0.95

gt=426 px  damaged=250 px  dice=0.621


In [9]:
#| export
def _perturb(mask: np.ndarray, sev: float, rng: np.random.Generator) -> np.ndarray:
    "Damage a 0/1 mask by `sev` in [0, 1]: shift, erode, add a false-positive circle, or lose it entirely."
    h, w = mask.shape
    reach = int(sev * 6)
    dx, dy = (int(v) for v in rng.integers(-reach, reach + 1, size=2))
    m = cv2.warpAffine(mask, np.float32([[1, 0, dx], [0, 1, dy]]), (w, h), flags=cv2.INTER_NEAREST)
    if rng.random() < sev:
        m = cv2.erode(m, np.ones((3, 3), np.uint8), iterations=1 + int(sev * 2))
    if rng.random() < sev:
        cv2.circle(m, (int(rng.integers(5, w - 5)), int(rng.integers(5, h - 5))), int(rng.integers(3, 8)), 1, -1)
    if rng.random() < sev * 0.4:
        m[:] = 0
    return m

def make_dummy_samples(
    n: int = 12,                      # number of samples
    size: Tuple[int, int] = (96, 128),  # (H, W)
    task: str = 'binary',             # 'binary' | 'multiclass' | 'multilabel'
    dtype: str = 'uint8',             # 'uint8' | 'uint16'
    seed: int = 0,                    # same seed gives the same samples
) -> List[Sample]:
    """Seeded synthetic samples with prediction errors of varying severity.

    Example:
        >>> [s.sample_id for s in make_dummy_samples(2)]
        ['dummy_000', 'dummy_001']
    """
    assert task in ('binary', 'multiclass', 'multilabel') and dtype in ('uint8', 'uint16')
    rng = np.random.default_rng(seed)
    h, w = size
    names = {'binary': None, 'multiclass': ['background', 'blob', 'bar'], 'multilabel': ['blob', 'bar']}[task]
    sev = rng.permutation(n) / max(n - 1, 1)
    out = []
    for i in range(n):
        blob, bar = np.zeros((h, w), np.uint8), np.zeros((h, w), np.uint8)
        for _ in range(int(rng.integers(1, 4))):
            cv2.ellipse(blob, (int(rng.integers(15, w - 15)), int(rng.integers(15, h - 15))),
                        (int(rng.integers(5, 14)), int(rng.integers(4, 10))), float(rng.integers(0, 180)), 0, 360, 1, -1)
        x0, y0 = int(rng.integers(5, w - 40)), int(rng.integers(5, h - 8))
        cv2.rectangle(bar, (x0, y0), (x0 + int(rng.integers(20, 35)), y0 + 2), 1, -1)
        gts = [blob] if task == 'binary' else [blob, bar]
        prs = [_perturb(m, sev[i], rng) for m in gts]
        if task == 'binary':
            gt, pr = gts[0], prs[0]
        elif task == 'multilabel':
            gt, pr = np.stack(gts), np.stack(prs)
        else:
            gt, pr = gts[0].copy(), prs[0].copy()
            gt[gts[1] > 0], pr[prs[1] > 0] = 2, 2
        signal = cv2.GaussianBlur(sum(gts).astype(np.float32), (0, 0), 1.5) * 0.35
        img = np.clip(0.25 + 0.15 * np.arange(w)[None, :] / w + rng.normal(0, 0.03, (h, w)) + signal, 0, 1)
        img = (img * 255).astype(np.uint8) if dtype == 'uint8' else (img * 60000).astype(np.uint16)
        out.append(Sample(f'dummy_{i:03d}', img, gt, pr, names, meta={'severity': round(float(sev[i]), 3)}))
    return out

In [10]:
# verify: three tasks, shapes, determinism, and that severity actually changes the score
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
for task, shape in [('binary', (96, 128)), ('multiclass', (96, 128)), ('multilabel', (2, 96, 128))]:
    ss = make_dummy_samples(6, task=task)
    assert ss[0].gt.shape == shape and ss[0].pred.shape == shape, (task, ss[0].gt.shape)
a, b = make_dummy_samples(6, seed=3), make_dummy_samples(6, seed=3)
assert all(np.array_equal(x.pred, y.pred) and np.array_equal(x.image, y.image) for x, y in zip(a, b))
assert make_dummy_samples(2, dtype='uint16')[0].image.dtype == np.uint16
ss = make_dummy_samples(6)
fig, ax = plt.subplots(2, 6, figsize=(14, 4.5))
for j, s in enumerate(ss):
    ax[0, j].imshow(s.image, cmap='gray'); ax[0, j].contour(s.gt, [0.5], colors='c'); ax[0, j].contour(s.pred, [0.5], colors='orange')
    ax[0, j].set_title(f"sev {s.meta['severity']}", fontsize=8); ax[0, j].axis('off')
    ax[1, j].imshow(s.gt.astype(int) * 2 + s.pred.astype(int)); ax[1, j].axis('off')
plt.tight_layout(); plt.show()

/tmp/ipykernel_345594/849913679.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [11]:
# test: determinism and layouts (nbdev-test runs this)
assert [s.sample_id for s in make_dummy_samples(3)] == ['dummy_000', 'dummy_001', 'dummy_002']
assert all(np.array_equal(x.gt, y.gt) for x, y in zip(make_dummy_samples(5, seed=1), make_dummy_samples(5, seed=1)))
assert not all(np.array_equal(x.pred, y.pred) for x, y in zip(make_dummy_samples(5, seed=1), make_dummy_samples(5, seed=2)))

## 3. Metrics through `SegMetrics`

`SegMetrics` takes logits. A stored 0/1 prediction becomes logits of +10 / -10, the same trick its own doctest uses, so the maths stays in one place.

Two traps handled here:

- A class that is **absent in both** ground truth and prediction has an undefined Dice. `SegMetrics` returns 0 (it adds a tiny constant), which would punish a correct "nothing there". We drop such classes from the per-sample score; if nothing is left the score is 1.0.
- For multiclass, background (label 0) does not count towards the score, matching `SegMetrics`' own mean.

*Experiment:* a hand-computed case. Ground truth has 4 pixels, prediction covers 2 of them plus 1 extra: tp=2, fp=1, fn=2, so Dice = 2*2/(2*2+1+2) = 4/7 and IoU = 2/5.

In [12]:
gt = np.zeros((4, 4), np.uint8); gt[0, :] = 1
pr = np.zeros((4, 4), np.uint8); pr[0, :2] = 1; pr[1, 0] = 1
m = SegMetrics(num_classes=1)
m.update(torch.from_numpy(pr.astype(np.float32) * 20 - 10)[None, None], torch.from_numpy(gt.astype(np.float32))[None, None])
out = m.compute()['per_class'][1]
print({k: round(v, 4) for k, v in out.items()})
assert abs(out['dice'] - 4 / 7) < 1e-4 and abs(out['iou'] - 2 / 5) < 1e-4

{'iou': 0.4, 'dice': 0.5714, 'precision': 0.6667, 'recall': 0.5}


In [13]:
#| export
def _feed(m: SegMetrics, st: _Stacks) -> None:
    "Update `m` with one sample; 0/1 predictions go in as +-10 logits."
    logits = torch.from_numpy(st.pred.astype(np.float32) * 20 - 10)[None]   # (1, K, H, W)
    gt = torch.from_numpy(st.gt.astype(np.float32))[None]
    m.update(logits, gt.argmax(1) if st.task == 'multiclass' else gt)

def _new_metrics(st: _Stacks) -> SegMetrics:
    return SegMetrics(num_classes=1 if st.task == 'binary' else len(st.names), task=st.task)

def _per_class(m: SegMetrics, st: _Stacks) -> Dict[str, Dict[str, float]]:
    "Per-class scores by name, skipping classes absent from both gt and prediction."
    keys = [1] if st.task == 'binary' else list(range(len(st.names)))
    per = m.compute()['per_class']
    out = {}
    for key, name in zip(keys, st.names):
        tp, fp, fn = m.tp[key], m.fp[key], m.fn[key]
        if tp + fp + fn == 0:
            continue
        out[name] = {**per[key], 'gt_px': tp + fn, 'pred_px': tp + fp}
    return out

def sample_metrics(s: Sample) -> Dict:
    """Per-class Dice, IoU, precision and recall for one sample, plus a single quality `score`.

    SegMetrics adds a tiny constant to avoid dividing by zero, so a perfect score is 0.9999995, not 1.

    The score is the mean Dice over classes present in gt or prediction (background excluded
    for multiclass), or 1.0 when nothing is present anywhere.

    Example:
        >>> g = np.zeros((4, 4), np.uint8); g[0] = 1
        >>> round(sample_metrics(Sample('a', g, g, g))['score'], 3)
        1.0
    """
    st = _class_stacks(s)
    m = _new_metrics(st)
    _feed(m, st)
    per = _per_class(m, st)
    skip = {st.names[0]} if st.task == 'multiclass' else set()
    dices = [v['dice'] for k, v in per.items() if k not in skip]
    return {'task': st.task, 'per_class': per, 'score': float(np.mean(dices)) if dices else 1.0}

def pooled_metrics(samples: Sequence[Sample]) -> Dict[str, float]:
    """Pixel counts pooled over all samples, as flat dotted keys such as `dice.void` and `iou.mean`.

    Pooled means large defects weigh more than the mean of per-sample scores would give them.
    Raises if samples disagree on task or class names.

    Example:
        >>> g = np.zeros((4, 4), np.uint8); g[0] = 1
        >>> pooled_metrics([Sample('a', g, g, g)])['dice.mean']
        1.0
    """
    m, first = None, None
    for s in samples:
        st = _class_stacks(s)
        if m is None:
            first, m = st, _new_metrics(st)
        elif (st.task, st.names) != (first.task, first.names):
            raise ValueError(f'{s.sample_id}: task/classes differ from the first sample')
        _feed(m, st)
    if m is None:
        return {}
    agg = m.compute()
    out = {}
    for name, v in _per_class(m, first).items():
        out[f'dice.{name}'], out[f'iou.{name}'] = round(v['dice'], 6), round(v['iou'], 6)
    out['dice.mean'], out['iou.mean'] = round(agg['dice'], 6), round(agg['iou'], 6)
    return out

In [14]:
# test: hand-computed values, absent classes, background handling, pooled vs per-sample
gt = np.zeros((4, 4), np.uint8); gt[0, :] = 1
pr = np.zeros((4, 4), np.uint8); pr[0, :2] = 1; pr[1, 0] = 1
img = np.zeros((4, 4), np.uint8)
r = sample_metrics(Sample('a', img, gt, pr))
assert abs(r['per_class']['foreground']['dice'] - 4 / 7) < 1e-4 and abs(r['per_class']['foreground']['iou'] - 2 / 5) < 1e-4
assert r['per_class']['foreground']['gt_px'] == 4 and r['per_class']['foreground']['pred_px'] == 3
assert sample_metrics(Sample('e', img, img, img))['score'] == 1.0          # nothing anywhere = perfect, not 0
assert sample_metrics(Sample('e', img, img, gt))['score'] == 0.0           # pure false positive = 0

lab = np.zeros((4, 4), np.uint8); lab[0, :2] = 1; lab[3, 2:] = 2
r = sample_metrics(Sample('m', img, lab, lab, ['bg', 'a', 'b']))
assert set(r['per_class']) == {'bg', 'a', 'b'} and abs(r['score'] - 1) < 1e-5

ml = np.stack([gt, np.zeros_like(gt)])
r = sample_metrics(Sample('l', img, ml, ml, ['x', 'y']))
assert list(r['per_class']) == ['x'] and abs(r['score'] - 1) < 1e-5                # 'y' is empty everywhere -> dropped

big = np.zeros((10, 10), np.uint8); big[:8] = 1                            # 80 px, perfect
small = np.zeros((10, 10), np.uint8); small[0, 0] = 1                      # 1 px, missed
p = pooled_metrics([Sample('big', np.zeros((10, 10)), big, big), Sample('small', np.zeros((10, 10)), small, np.zeros_like(small))])
assert p['dice.foreground'] > 0.97 and p['dice.mean'] == p['dice.foreground']   # pooled is dominated by the big one
try: pooled_metrics([Sample('a', img, gt, gt), Sample('b', img, ml, ml)])
except ValueError as e: print('ok:', e)
else: raise AssertionError

ok: b: task/classes differ from the first sample


## 4. Display: percentile window and thumbnails

A browser shows 8 bits. A 16-bit X-ray squeezed linearly into 8 bits is often a grey smear, so we map the 1st to 99.5th percentile to 0-255 (the window). `uint8` images pass through untouched. Three-channel images are shown as RGB; any other channel count shows the first channel.

Thumbnails are JPEG (quality ~85, long edge ~512, never upscaled), base64-embedded. Masks never go through JPEG; they become SVG in the next step.

*Experiment:* a low-contrast 16-bit image, before and after windowing.

In [15]:
rng = np.random.default_rng(0)
raw = (rng.normal(30000, 400, (64, 64))).clip(0, 65535).astype(np.uint16); raw[20:40, 20:40] += 1500; raw[0, 0] = 65535   # one hot pixel
naive = ((raw.astype(np.float32) - raw.min()) / (raw.max() - raw.min()) * 255).astype(np.uint8)   # plain min/max stretch
lo, hi = np.percentile(raw.astype(np.float32), [1, 99.5])
win = (np.clip((raw.astype(np.float32) - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
bulk = np.ones(raw.shape, bool); bulk[0, 0] = False                                                 # everything except the hot pixel
print('grey levels used by the real picture: min/max stretch', np.ptp(naive[bulk]), '| percentile window', np.ptp(win[bulk]))
assert np.ptp(naive[bulk]) < 40 and np.ptp(win[bulk]) > 200       # one hot pixel flattens a plain stretch; the window does not care

grey levels used by the real picture: min/max stretch 28 | percentile window 255


In [16]:
#| export
def _pick_channels(image: np.ndarray) -> np.ndarray:
    "RGB stays RGB; any other channel count shows the first channel."
    a = np.asarray(image)
    if a.ndim == 3 and a.shape[2] != 3:
        a = a[..., 0]
    return a

def to_display_uint8(
    image: np.ndarray,                  # (H, W) or (H, W, C), any dtype
    p_lo: float = 1.0,                  # percentile mapped to 0
    p_hi: float = 99.5,                 # percentile mapped to 255
) -> np.ndarray:
    """Map an image to 8 bits for the screen; `uint8` is returned unchanged.

    Example:
        >>> to_display_uint8(np.array([[0, 1000, 2000]], np.uint16), 0, 100).tolist()
        [[0, 128, 255]]
    """
    a = _pick_channels(image)
    if a.dtype == np.uint8:
        return a
    a = a.astype(np.float32)
    lo, hi = np.percentile(a, [p_lo, p_hi])
    if hi <= lo:
        hi = lo + 1.0
    return np.round(np.clip((a - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

def thumbnail_b64(
    image: np.ndarray,
    long_edge: int = 512,               # never upscaled
    quality: int = 85,                  # JPEG quality
    window: Tuple[float, float] = (1.0, 99.5),
) -> str:
    "Base64 JPEG thumbnail of `image`, ready for a `data:image/jpeg;base64,` URI."
    disp = to_display_uint8(image, *window)
    h, w = disp.shape[:2]
    s = long_edge / max(h, w)
    if s < 1:
        disp = cv2.resize(disp, (max(1, round(w * s)), max(1, round(h * s))), interpolation=cv2.INTER_AREA)
    bgr = disp[..., ::-1] if disp.ndim == 3 else disp
    ok, buf = cv2.imencode('.jpg', np.ascontiguousarray(bgr), [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        raise OSError('JPEG encoding failed')
    return base64.b64encode(buf.tobytes()).decode('ascii')

def _write_full_png(path: Path, image: np.ndarray, window: Tuple[float, float]) -> bool:
    "Write the full-resolution file; True when raw values were kept (uint8/uint16), False for a windowed copy."
    a = _pick_channels(image)
    raw = a.dtype in (np.uint8, np.uint16)
    out = a if raw else to_display_uint8(a, *window)
    if out.ndim == 3:
        out = out[..., ::-1]
    if not cv2.imwrite(str(path), np.ascontiguousarray(out)):
        raise OSError(f'could not write {path}')
    return raw

In [17]:
# test: uint8 untouched, 16-bit windowed, thumbnail size, raw values survive the full-resolution file
import tempfile
u8 = np.arange(12, dtype=np.uint8).reshape(3, 4)
assert to_display_uint8(u8) is u8 or np.array_equal(to_display_uint8(u8), u8)
assert to_display_uint8(raw).max() - to_display_uint8(raw).min() > 200
assert to_display_uint8(np.zeros((3, 3, 4), np.uint16)).shape == (3, 3)           # 4 channels -> first
assert to_display_uint8(np.zeros((3, 3, 3), np.uint16)).shape == (3, 3, 3)        # RGB kept

big = np.zeros((1200, 600), np.uint16)
jpg = cv2.imdecode(np.frombuffer(base64.b64decode(thumbnail_b64(big, long_edge=512)), np.uint8), cv2.IMREAD_UNCHANGED)
assert jpg.shape == (512, 256), jpg.shape                                          # long edge 512, aspect kept
small = np.zeros((40, 30), np.uint8)
assert cv2.imdecode(np.frombuffer(base64.b64decode(thumbnail_b64(small)), np.uint8), cv2.IMREAD_UNCHANGED).shape == (40, 30)   # no upscale

with tempfile.TemporaryDirectory() as d:
    assert _write_full_png(Path(d) / 'a.png', raw, (1, 99.5)) is True
    assert np.array_equal(cv2.imread(str(Path(d) / 'a.png'), cv2.IMREAD_UNCHANGED), raw)   # 16-bit values identical
    assert _write_full_png(Path(d) / 'b.png', raw.astype(np.float32), (1, 99.5)) is False

## 5. Masks as SVG contours

The overlay must stay exact, so masks are drawn as vector outlines in **original pixel coordinates**; the browser scales them with the thumbnail. `cv2.findContours` gives the outlines, `RETR_TREE` keeps holes and islands, and one SVG path with `fill-rule: evenodd` draws all of them correctly. `CHAIN_APPROX_SIMPLE` drops only points on straight runs, so it is exact and smaller than keeping every pixel.

"Exact" needs a test, not a belief. The check below draws the path back into pixels (fill every sub-path, XOR them so nested holes cancel, then put back the outline pixels, which are always foreground) and compares with the source mask. First on 300 random masks with holes and diagonal contacts (done in the planning experiment: 0 mismatches), then in the test cell below.

In [18]:
#| export
def mask_to_svg_path(mask: np.ndarray) -> str:
    """SVG path data tracing every region of a binary mask, holes and islands included.

    Coordinates are pixel indices (x y), so a `viewBox` of `0 0 W H` lines the path up with the image.
    Returns an empty string for an empty mask.

    Example:
        >>> mask_to_svg_path(np.pad(np.ones((1, 1), np.uint8), 1))
        'M1 1Z'
    """
    m = (np.asarray(mask) > 0).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    return ''.join('M' + ' '.join(f'{x} {y}' for x, y in c.reshape(-1, 2)) + 'Z' for c in contours)

In [19]:
def _path_to_mask(path: str, shape) -> np.ndarray:
    "Test helper: rasterise path data back to a mask (even-odd via XOR, outline pixels restored)."
    acc, outline = np.zeros(shape, np.uint8), np.zeros(shape, np.uint8)
    for body in re.findall(r'M([^Z]*)Z', path):
        pts = np.array(body.split(), dtype=np.int32).reshape(-1, 2)
        layer = np.zeros(shape, np.uint8); cv2.fillPoly(layer, [pts], 1); acc ^= layer
        cv2.polylines(outline, [pts], True, 1, 1); outline[pts[:, 1], pts[:, 0]] = 1
    return acc | outline

# test: path round-trips to the exact mask, for hand-made and random masks
ring = np.zeros((20, 20), np.uint8); ring[2:18, 2:18] = 1; ring[6:14, 6:14] = 0; ring[9:11, 9:11] = 1   # square, hole, island in the hole
assert np.array_equal(_path_to_mask(mask_to_svg_path(ring), ring.shape), ring)
assert mask_to_svg_path(np.zeros((5, 5))) == ''
one = np.zeros((5, 5), np.uint8); one[2, 3] = 1
assert np.array_equal(_path_to_mask(mask_to_svg_path(one), one.shape), one)                           # single pixel
rng = np.random.default_rng(1)
for t in range(200):
    m = (rng.random((24, 32)) > rng.choice([0.35, 0.5, 0.65])).astype(np.uint8)
    assert np.array_equal(_path_to_mask(mask_to_svg_path(m), m.shape), m), t
for s in make_dummy_samples(6):
    assert np.array_equal(_path_to_mask(mask_to_svg_path(s.pred), s.pred.shape), s.pred)

## 6. Which samples to show

Showing every sample of a validation set makes a page nobody reads. We show three small groups: the **worst** (lowest score), the **best**, and a **random** draw from the rest, so the reviewer sees failures, successes and typical cases. The score is "higher is better" (`Sample.score` if given, else mean Dice). A fixed seed makes the page reproducible; a sample is never in two groups.

*Experiment:* the group logic on plain numbers.

In [20]:
scores = np.array([0.9, 0.1, 0.5, 0.7, 0.3, 0.95, 0.2, 0.6])
order = np.argsort(scores, kind='stable')
worst = order[:2].tolist(); best = [i for i in order[::-1] if i not in worst][:2]
rest = [i for i in range(len(scores)) if i not in worst + best]
rnd = sorted(np.random.default_rng(0).choice(rest, size=2, replace=False).tolist())
print('worst', worst, 'best', best, 'random', rnd)
assert worst == [1, 6] and best == [5, 0] and not set(rnd) & set(worst + best)

worst [1, 6] best [np.int64(5), np.int64(0)] random [4, 7]


In [21]:
#| export
def select_samples(
    scores: Sequence[float],   # quality per sample, higher is better
    n_worst: int = 4,
    n_random: int = 4,
    n_best: int = 4,
    seed: int = 0,             # same seed, same random group
) -> List[Tuple[int, str]]:
    """Pick sample indices for the groups `worst`, `random`, `best`; no index appears twice.

    Worst comes first (lowest score first), then random (in dataset order), then best (highest first).

    Example:
        >>> select_samples([0.9, 0.1, 0.5], 1, 0, 1)
        [(1, 'worst'), (0, 'best')]
    """
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(scores, kind='stable')
    worst = [int(i) for i in order[:n_worst]]
    taken = set(worst)
    best = [int(i) for i in order[::-1] if int(i) not in taken][:n_best]
    taken |= set(best)
    rest = [i for i in range(len(scores)) if i not in taken]
    k = min(n_random, len(rest))
    rnd = sorted(int(i) for i in np.random.default_rng(seed).choice(rest, size=k, replace=False)) if k else []
    return [(i, 'worst') for i in worst] + [(i, 'random') for i in rnd] + [(i, 'best') for i in best]

In [22]:
# test: determinism, disjoint groups, small inputs, ordering
sc = [0.9, 0.1, 0.5, 0.7, 0.3, 0.95, 0.2, 0.6]
a = select_samples(sc, 2, 2, 2, seed=0)
assert a == select_samples(sc, 2, 2, 2, seed=0)                                   # same seed, same result
assert [i for i, g in a if g == 'worst'] == [1, 6] and [i for i, g in a if g == 'best'] == [5, 0]
assert len({i for i, _ in a}) == len(a) == 6
assert any(select_samples(sc, 0, 3, 0, seed=s) != select_samples(sc, 0, 3, 0, seed=0) for s in range(1, 20))
assert len(select_samples([0.5, 0.4], 4, 4, 4)) == 2                              # fewer samples than requested
assert select_samples([], 2, 2, 2) == []

## 7. Sections: the plug-in interface

A **section** is a function from a `ReportContext` to a `SectionOutput`. The context carries the ticket id, output folder, samples, seed and config, plus a small helper to write files under `assets/`. A section returns a title, an HTML fragment, a status, and **flat metrics with dotted keys** (`dice.void`); the report merges them into `report.json`, and gates (ticket #52) read from that dict.

Sections register by name, so `build_report(..., sections=['overlay'])` works, and another map can pass its own function directly without editing this module.

In [23]:
#| export
@dataclass
class SectionOutput:
    "What a section hands back to the report."
    title: str
    html: str                                     # body fragment; no external URLs
    status: str = 'info'                          # 'info' | 'pass' | 'fail'
    metrics: Dict[str, float] = field(default_factory=dict)   # flat dotted keys, e.g. {'dice.void': 0.83}

@dataclass
class ReportContext:
    "Everything a section needs, plus a place to leave notes for `report.json`."
    ticket_id: str
    out_dir: Path
    samples: List[Sample]
    seed: int = 0
    cfg: Dict = field(default_factory=dict)
    state: Dict = field(default_factory=dict)     # JSON-serialisable notes, written to report.json

    def asset_path(self, *parts: str) -> Path:
        "Path under `out_dir/assets/`, with parent folders created."
        p = Path(self.out_dir, 'assets', *parts)
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def rel(self, path) -> str:
        "Path relative to the report folder, with forward slashes, for use in HTML."
        return Path(path).relative_to(self.out_dir).as_posix()

SECTIONS: Dict[str, Callable[[ReportContext], SectionOutput]] = {}

def register_section(name: str):
    """Decorator that adds a section function to the registry under `name`.

    Example:
        >>> @register_section('demo')
        ... def demo(ctx): return SectionOutput('Demo', '<p>hi</p>')
        >>> 'demo' in SECTIONS
        True
    """
    def deco(fn):
        SECTIONS[name] = fn
        return fn
    return deco

DEFAULT_CFG = dict(
    n_worst=4, n_random=4, n_best=4,      # samples per group in the overlay section
    thumb_edge=512, jpeg_quality=85,      # thumbnail long edge (px) and JPEG quality
    window=(1.0, 99.5),                   # display percentiles for non-uint8 images
)

In [24]:
# test: registry and context helpers
import tempfile
@register_section('_t')
def _t(ctx): return SectionOutput('T', '<p>x</p>')
assert SECTIONS['_t'] is _t
del SECTIONS['_t']
with tempfile.TemporaryDirectory() as d:
    ctx = ReportContext('T-1', Path(d), [])
    p = ctx.asset_path('s1', 'full.png')
    assert p.parent.is_dir() and ctx.rel(p) == 'assets/s1/full.png'
    assert not (Path(d) / 'assets' / 's1' / 'full.png').exists()                # a path is handed out, nothing is written

## 8. The overlay section

One card per shown sample:

- the thumbnail with an SVG on top (viewBox = original pixels), four layers: **image**, **ground truth** (cyan outline), **prediction** (orange outline), **errors** (false positive red, false negative green, the same colours `save_error_poster` uses for its difference column);
- a class selector when there is more than one class, and layer check boxes;
- a per-class table (Dice, IoU, precision, recall);
- a link to the full-resolution file.

Each card carries `data-sample-id`, `data-w`, `data-h` (original size). Ticket #51 turns a click into original-pixel coordinates from exactly these.

For multiclass tasks the background label is not drawn (it would cover the image) but it is in the table.

A report of 60 samples has to stay small, so the section writes only three things per sample: the JPEG thumbnail inside the page, paths for the shown classes, and one full-resolution file in `assets/`.

In [25]:
#| export
def _safe_id(sample_id: str) -> str:
    "Filesystem-safe form of a sample id."
    return re.sub(r'[^A-Za-z0-9._-]', '_', str(sample_id)).lstrip('.') or '_'

def _check_samples(samples: Sequence[Sample]) -> None:
    "Reject duplicate ids, and ids that collide once made filesystem-safe."
    seen: Dict[str, str] = {}
    for s in samples:
        safe = _safe_id(s.sample_id)
        if safe in seen:
            raise ValueError(f'sample ids {seen[safe]!r} and {s.sample_id!r} map to the same asset folder {safe!r}')
        seen[safe] = s.sample_id

def _layer_paths(st: _Stacks, k: int) -> Dict[str, str]:
    "SVG path data for class `k`: gt, pred, and the two error layers."
    g, p = st.gt[k], st.pred[k]
    return {'gt': mask_to_svg_path(g), 'pred': mask_to_svg_path(p),
            'fp': mask_to_svg_path(p & ~g), 'fn': mask_to_svg_path(g & ~p)}

def _fmt(v: float) -> str:
    return f'{v:.3f}'

def _card(ctx: ReportContext, s: Sample, group: str, met: Dict, score: float) -> str:
    st = _class_stacks(s)
    h, w = st.gt.shape[1:]
    full = ctx.asset_path(_safe_id(s.sample_id), 'full.png')
    raw_kept = _write_full_png(full, s.image, tuple(ctx.cfg['window']))
    first = 1 if st.task == 'multiclass' else 0           # multiclass background is not drawn
    groups = []
    for k in range(first, len(st.names)):
        paths = _layer_paths(st, k)
        shapes = ''.join(f'<path class="{layer}" d="{d}"/>' for layer, d in paths.items() if d)
        if shapes:
            groups.append(f'<g data-class="{esc(st.names[k], quote=True)}">{shapes}</g>')
    drawn = [st.names[k] for k in range(first, len(st.names))]
    select = ''
    if len(drawn) > 1:
        opts = '<option value="__all__">all classes</option>' + ''.join(f'<option>{esc(n)}</option>' for n in drawn)
        select = f'<label>class <select class="cls">{opts}</select></label>'
    rows = ''.join(
        f'<tr><td>{esc(n)}</td><td>{_fmt(v["dice"])}</td><td>{_fmt(v["iou"])}</td>'
        f'<td>{_fmt(v["precision"])}</td><td>{_fmt(v["recall"])}</td></tr>' for n, v in met['per_class'].items())
    table = (f'<table><thead><tr><th>class</th><th>Dice</th><th>IoU</th><th>prec.</th><th>recall</th></tr></thead><tbody>{rows}</tbody></table>'
             if rows else '<p class="note">No foreground in ground truth or prediction.</p>')
    note = '' if raw_kept else ' <span class="note">(windowed copy, not raw values)</span>'
    img = thumbnail_b64(s.image, ctx.cfg['thumb_edge'], ctx.cfg['jpeg_quality'], tuple(ctx.cfg['window']))
    return (
        f'<article class="card" data-sample-id="{esc(s.sample_id, quote=True)}" data-w="{w}" data-h="{h}">'
        f'<header><strong>{esc(s.sample_id)}</strong><span class="badge {group}">{group}</span>'
        f'<span class="note">score {_fmt(score)}</span></header>'
        f'<div class="stage" style="aspect-ratio:{w}/{h}">'
        f'<img alt="{esc(s.sample_id, quote=True)}" src="data:image/jpeg;base64,{img}">'
        f'<svg viewBox="0 0 {w} {h}" preserveAspectRatio="none" fill-rule="evenodd">{"".join(groups)}</svg></div>'
        f'<div class="controls">'
        + ''.join(f'<label><input type="checkbox" data-layer="{l}" checked> {t}</label>'
                  for l, t in [('image', 'image'), ('gt', 'ground truth'), ('pred', 'prediction'), ('err', 'errors')])
        + f'{select}</div>{table}'
        f'<a class="note" href="{esc(ctx.rel(full), quote=True)}">full resolution ({w}x{h}){note}</a></article>')

@register_section('overlay')
def overlay_section(ctx: ReportContext) -> SectionOutput:
    """Ground truth vs prediction vs errors for the worst, random and best samples.

    Metrics returned are pooled over *all* samples, not only the ones shown.
    """
    mets = [sample_metrics(s) for s in ctx.samples]
    scores = [m['score'] if s.score is None else s.score for s, m in zip(ctx.samples, mets)]
    c = ctx.cfg
    picked = select_samples(scores, c['n_worst'], c['n_random'], c['n_best'], ctx.seed)
    ctx.state['overlay'] = {g: [ctx.samples[i].sample_id for i, gg in picked if gg == g] for g in ('worst', 'random', 'best')}
    blurbs = {'worst': 'Lowest score first.', 'random': 'Random draw from the rest (fixed seed).', 'best': 'Highest score first.'}
    body = [f'<p class="note">Showing {len(picked)} of {len(ctx.samples)} samples. Outline colours: '
            f'<span class="key gt">ground truth</span> <span class="key pred">prediction</span>. '
            f'Errors: <span class="key fp">false positive</span> <span class="key fn">false negative</span>.</p>']
    for g in ('worst', 'random', 'best'):
        cards = ''.join(_card(ctx, ctx.samples[i], g, mets[i], scores[i]) for i, gg in picked if gg == g)
        if cards:
            body.append(f'<h3>{g.capitalize()} <span class="note">{blurbs[g]}</span></h3><div class="grid">{cards}</div>')
    return SectionOutput('Overlay: ground truth vs prediction', ''.join(body), 'info', pooled_metrics(ctx.samples))

## 9. The page and `build_report`

The page is plain strings: a small CSS block (light and dark), the sections, and about twenty lines of JavaScript that only toggle visibility, with no network and no libraries. Dark mode follows the system setting.

*Experiment (done outside this notebook, by looking at it):* build a report from dummy data, open it in a browser, and check layout, toggles and the class selector. The test cell below checks the structure automatically (layers, data attributes, no external URLs, no JPEG masks, size); the visual check is by eye.

In [26]:
#| export
_CSS = """
:root{--bg:#fff;--fg:#1b1d21;--muted:#5f6670;--card:#f4f5f7;--line:#d6d9de;--gt:#0097a7;--pred:#ef6c00;--fp:#d32f2f;--fn:#2e7d32;--accent:#3b5bdb}
@media (prefers-color-scheme: dark){:root{--bg:#14161a;--fg:#e9eaec;--muted:#a0a6ae;--card:#1d2026;--line:#343944;--gt:#4dd0e1;--pred:#ffb74d;--fp:#ef5350;--fn:#66bb6a;--accent:#8ea2ff}}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--fg);font:15px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif}
.top,main{max-width:1200px;margin:0 auto;padding:16px}
main{padding-top:0} main>section:first-child h2{margin-top:8px}
h1{font-size:1.4rem;margin:0 0 4px} h2{font-size:1.15rem;margin:28px 0 8px;border-bottom:1px solid var(--line);padding-bottom:4px} h3{font-size:1rem;margin:20px 0 8px}
.note{color:var(--muted);font-size:.85rem;font-weight:400}
.grid{display:grid;grid-template-columns:repeat(auto-fill,minmax(300px,1fr));gap:14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:8px;padding:10px;display:flex;flex-direction:column;gap:8px;min-width:0}
.card header{display:flex;gap:8px;align-items:baseline;flex-wrap:wrap}
.badge{font-size:.72rem;padding:1px 7px;border-radius:10px;border:1px solid var(--line);text-transform:uppercase;letter-spacing:.03em}
.badge.worst{color:var(--fp)} .badge.best{color:var(--fn)} .badge.random{color:var(--muted)}
.stage{position:relative;width:100%;background:#000;border-radius:4px;overflow:hidden}
.stage img,.stage svg{position:absolute;inset:0;width:100%;height:100%;display:block}
.stage svg path{vector-effect:non-scaling-stroke;stroke-width:1.5}
.gt{fill:var(--gt);fill-opacity:.12;stroke:var(--gt)} .pred{fill:var(--pred);fill-opacity:.12;stroke:var(--pred)}
.fp{fill:var(--fp);fill-opacity:.55;stroke:none} .fn{fill:var(--fn);fill-opacity:.55;stroke:none}
.stage.hide-image img{visibility:hidden} .stage.hide-gt .gt{display:none} .stage.hide-pred .pred{display:none} .stage.hide-err .fp,.stage.hide-err .fn{display:none}
.key{padding:0 6px;border-radius:3px;border:1px solid currentColor} .key.gt{color:var(--gt)} .key.pred{color:var(--pred)} .key.fp{color:var(--fp)} .key.fn{color:var(--fn)}
.controls{display:flex;flex-wrap:wrap;gap:4px 12px;font-size:.85rem} .controls select{font:inherit}
table{border-collapse:collapse;width:100%;font-size:.82rem;font-variant-numeric:tabular-nums}
th,td{text-align:right;padding:2px 6px;border-bottom:1px solid var(--line)} th:first-child,td:first-child{text-align:left}
a{color:var(--accent)}
"""

_JS = """
document.querySelectorAll('.card').forEach(function(card){
  var stage=card.querySelector('.stage');
  function apply(){
    card.querySelectorAll('input[data-layer]').forEach(function(cb){stage.classList.toggle('hide-'+cb.dataset.layer,!cb.checked);});
    var sel=card.querySelector('select.cls'), cls=sel?sel.value:'__all__';
    card.querySelectorAll('g[data-class]').forEach(function(g){g.style.display=(cls==='__all__'||g.dataset.class===cls)?'':'none';});
  }
  card.addEventListener('change',apply); apply();
});
"""

def build_report(
    ticket_id: str,                         # letters, digits, '.', '_' and '-' only (used in file names)
    out_dir,                                # report folder, created if missing; one folder per ticket
    samples,                                # iterable of `Sample`
    sections=('overlay',),                  # registered names, or section functions
    seed: int = 0,                          # fixes which random samples are shown
    cfg: Optional[Dict] = None,             # overrides for `DEFAULT_CFG`
) -> Path:
    """Write `report.html` (one self-contained page) and `report.json`, return the path to the page.

    Example:
        >>> import tempfile
        >>> p = build_report('T-1', tempfile.mkdtemp(), make_dummy_samples(6))
        >>> p.name
        'report.html'
    """
    if not re.fullmatch(r'[A-Za-z0-9._-]+', ticket_id):
        raise ValueError(f'ticket_id {ticket_id!r} may only contain letters, digits, ".", "_" and "-"')
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    samples = list(samples)
    _check_samples(samples)
    ctx = ReportContext(ticket_id, out_dir, samples, seed, {**DEFAULT_CFG, **(cfg or {})})
    outputs: List[Tuple[str, SectionOutput]] = []
    for sec in sections:
        fn = SECTIONS.get(sec) if isinstance(sec, str) else sec
        if fn is None:
            raise ValueError(f'unknown section {sec!r}; available: {sorted(SECTIONS)}')
        outputs.append((sec if isinstance(sec, str) else getattr(sec, '__name__', 'section'), fn(ctx)))
    body = ''.join(f'<section id="{esc(n, quote=True)}"><h2>{esc(o.title)}</h2>{o.html}</section>' for n, o in outputs)
    title = f'Review report {ticket_id}'
    page = ('<!doctype html><html lang="en"><head><meta charset="utf-8">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">'
            f'<title>{esc(title)}</title><style>{_CSS}</style></head><body>'
            f'<div class="top"><h1>{esc(title)}</h1><p class="note">{len(samples)} samples, seed {seed}</p></div>'
            f'<main>{body}</main><script>{_JS}</script></body></html>')
    path = out_dir / 'report.html'
    path.write_text(page, encoding='utf-8')
    metrics = {k: v for _, o in outputs for k, v in o.metrics.items()}
    summary = dict(schema_version=1, ticket_id=ticket_id, seed=seed, n_samples=len(samples),
                   sections=[dict(name=n, title=o.title, status=o.status) for n, o in outputs],
                   metrics=metrics, **ctx.state)
    (out_dir / 'report.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return path

In [27]:
# test: structure for all three tasks and both bit depths
import tempfile
for task, dtype in [('binary', 'uint8'), ('multiclass', 'uint8'), ('multilabel', 'uint16')]:
    with tempfile.TemporaryDirectory() as d:
        ss = make_dummy_samples(12, task=task, dtype=dtype)
        page = build_report('T-1', d, ss, seed=5).read_text()
        info = json.loads((Path(d) / 'report.json').read_text())
        shown = sum(info['overlay'].values(), [])
        assert page.count('class="card"') == len(shown) == 12 and len(set(shown)) == 12
        assert page.count('data:image/jpeg;base64,') == 12                       # JPEG only for photos
        assert 'src="http' not in page and 'href="http' not in page and 'url(http' not in page and '@import' not in page
        assert list(Path(d, 'assets').rglob('*.jpg')) == [] and len(list(Path(d, 'assets').rglob('full.png'))) == 12
        assert 'class="gt"' in page and 'class="pred"' in page and 'class="fn"' in page
        assert ('select class="cls"' in page) == (task != 'binary')
        for s in ss:
            h, w = s.gt.shape[-2:]
            assert f'data-sample-id="{s.sample_id}" data-w="{w}" data-h="{h}"' in page     # what ticket #51 relies on
        assert info['schema_version'] == 1 and 'dice.mean' in info['metrics'] and info['ticket_id'] == 'T-1'
        if dtype == 'uint16':
            full = cv2.imread(str(Path(d, 'assets', ss[0].sample_id, 'full.png')), cv2.IMREAD_UNCHANGED)
            assert full.dtype == np.uint16 and np.array_equal(full, ss[0].image)          # raw 16-bit values kept

# test: same seed -> same page choice; different seed -> maybe different random group; ids and sections validated
ss = make_dummy_samples(20)
with tempfile.TemporaryDirectory() as d1, tempfile.TemporaryDirectory() as d2:
    build_report('T', d1, ss, seed=1); build_report('T', d2, ss, seed=1)
    assert json.loads(Path(d1, 'report.json').read_text()) == json.loads(Path(d2, 'report.json').read_text())
    assert Path(d1, 'report.html').read_text() == Path(d2, 'report.html').read_text()
    assert len(json.loads(Path(d1, 'report.json').read_text())['overlay']['worst']) == 4
for bad, exc in [(dict(ticket_id='../x'), 'ticket_id'), (dict(ticket_id='T', sections=['nope']), 'unknown section')]:
    try: build_report(**{'out_dir': tempfile.mkdtemp(), 'samples': ss[:2], **bad})
    except ValueError as e: assert exc in str(e)
    else: raise AssertionError(exc)
dup = [ss[0], Sample(ss[0].sample_id, ss[0].image, ss[0].gt, ss[0].pred)]
try: build_report('T', tempfile.mkdtemp(), dup)
except ValueError as e: print('ok:', e)
else: raise AssertionError

# test: a client-supplied section function works without touching the registry
def extra(ctx): return SectionOutput('Extra', '<p>hello</p>', 'pass', {'extra.value': 1.0})
with tempfile.TemporaryDirectory() as d:
    build_report('T', d, ss[:3], sections=['overlay', extra])
    assert json.loads(Path(d, 'report.json').read_text())['metrics']['extra.value'] == 1.0
    assert 'hello' in Path(d, 'report.html').read_text()

ok: sample ids 'dummy_000' and 'dummy_000' map to the same asset folder 'dummy_000'


In [28]:
# test: size budget. 60 samples of 16-bit 512x512 must stay well under 25 MB for the page; assets are separate
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    ss = make_dummy_samples(60, size=(512, 512), dtype='uint16')
    page = build_report('T-size', d, ss, cfg=dict(n_worst=20, n_random=20, n_best=20))
    mb = os.path.getsize(page) / 1e6
    print(f'report.html {mb:.2f} MB for 60 samples')
    assert mb < 25

report.html 10.45 MB for 60 samples


## 10. The real-data path: model + dataset to `Sample`s

Everything above works on stored arrays. `samples_from_dataset` is the bridge from a trained model and the library's datasets: it runs `predict` + `decode` (the public inference functions, so activation and thresholds follow the same rules as everywhere else) and builds a `Sample` per item.

**Raw images.** The library's datasets read channels as 8-bit grayscale and return normalised tensors, so a 16-bit file has already lost its range by the time it is a tensor. Pass `raw_image_fn(sample_id) -> ndarray` to re-read the original file (for example with `cv2.IMREAD_UNCHANGED`); file naming differs between projects, so this is a hook and not a guess. Without it the page shows the model input (first channel, or RGB for three channels) and says so in the full-resolution note.

*Experiment:* a tiny stand-in model and dataset, to check shapes and the three tasks.

In [29]:
import torch.nn as nn
h, w = 16, 20
mask = torch.zeros(h, w).long(); mask[4:10, 5:12] = 1
logits = (mask.float() * 20 - 10)[None]                       # (1, H, W): what a perfect C=1 model would output
assert (torch.sigmoid(logits[0]) >= 0.5).long().equal(mask)   # sigmoid then threshold recovers the mask
print('ok:', int(mask.sum()), 'foreground px recovered')

ok: 42 foreground px recovered


In [30]:
#| export
@torch.no_grad()
def samples_from_dataset(
    model,                                   # trained model returning (B, C, H, W) logits
    dataset,                                 # items are {'image', 'mask', 'sample_id'}
    device: torch.device,
    task: Optional[str] = None,              # 'binary' | 'multiclass' | 'multilabel'; None infers from channels
    class_names: Optional[List[str]] = None, # see the `Sample` layout table
    threshold=0.5,                           # decision threshold (float, or per-channel list for multilabel)
    raw_image_fn: Optional[Callable[[str], np.ndarray]] = None,  # re-read the original file by sample id
    n: Optional[int] = None,                 # use the first n items; None = all
) -> List[Sample]:
    """Run `model` over `dataset` once and return a `Sample` per item.

    Example:
        >>> samples = samples_from_dataset(model, val_ds, device, class_names=['void'])  # doctest: +SKIP
    """
    model.eval()
    out = []
    for i in range(min(n or len(dataset), len(dataset))):
        item = dataset[i]
        sid = str(item['sample_id'])
        pred = decode(predict(model, item['image'], device, task=task), threshold, task).cpu().numpy()
        gt = item['mask'].cpu().numpy()
        gt = (gt > 0.5).astype(np.uint8) if gt.ndim == 3 else gt.astype(np.int64)
        if raw_image_fn is not None:
            image = raw_image_fn(sid)
        else:
            t = item['image'].cpu().numpy()                                  # (C, H, W)
            image = t.transpose(1, 2, 0) if t.shape[0] == 3 else t[0]
        out.append(Sample(sid, image, gt, pred.astype(gt.dtype), class_names))
    return out

In [31]:
# test: the adapter on a stand-in model/dataset, all three tasks, and the raw-image hook
class _DS:
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]
class _Fixed(nn.Module):
    "Ignores its input and returns the logits of the item being predicted."
    def __init__(self, items): super().__init__(); self.items = items; self.k = 0
    def forward(self, x): out = self.items[self.k]['logits']; self.k = (self.k + 1) % len(self.items); return out[None]

h, w = 16, 20
m = torch.zeros(h, w).long(); m[4:10, 5:12] = 1
items = [{'image': torch.rand(1, h, w), 'mask': m, 'sample_id': 'a', 'logits': (m.float() * 20 - 10)[None]}]
ss = samples_from_dataset(_Fixed(items), _DS(items), torch.device('cpu'), class_names=['void'])
assert ss[0].sample_id == 'a' and ss[0].gt.shape == (h, w) and np.array_equal(ss[0].pred, m.numpy())
assert ss[0].image.shape == (h, w) and abs(sample_metrics(ss[0])['score'] - 1) < 1e-5

ml = torch.stack([m, torch.zeros_like(m)]).float()
items = [{'image': torch.rand(1, h, w), 'mask': ml, 'sample_id': 'b', 'logits': ml * 20 - 10}]
ss = samples_from_dataset(_Fixed(items), _DS(items), torch.device('cpu'), task='multilabel', class_names=['x', 'y'])
assert ss[0].gt.shape == (2, h, w) and np.array_equal(ss[0].pred, ml.numpy().astype(ss[0].pred.dtype))

lab = m.clone(); lab[12:15, 2:6] = 2
lg = torch.nn.functional.one_hot(lab, 3).permute(2, 0, 1).float() * 20 - 10
items = [{'image': torch.rand(3, h, w), 'mask': lab, 'sample_id': 'c', 'logits': lg}]
raw = np.full((h, w), 40000, np.uint16)
ss = samples_from_dataset(_Fixed(items), _DS(items), torch.device('cpu'), class_names=['bg', 'x', 'y'], raw_image_fn=lambda sid: raw)
assert ss[0].image is raw and np.array_equal(ss[0].pred, lab.numpy())                    # multiclass argmax, raw hook used
ss = samples_from_dataset(_Fixed(items), _DS(items), torch.device('cpu'), class_names=['bg', 'x', 'y'])
assert ss[0].image.shape == (h, w, 3)                                                    # three channels -> RGB
import tempfile
with tempfile.TemporaryDirectory() as d:
    build_report('T-adapter', d, ss)

### On real data

The synthetic tests above guard the code. The point of the report is real images, so the same call is run on a real validation set. This cell needs a trained checkpoint and a data path, so it does not run in `nbdev-test` (`eval: false`). Fill in the paths from your config and run it by hand.

In [33]:
#| eval: false
# Machine-specific: run by hand. Paths come from your Hydra config; real data is never committed.
import cv2, torch
from segmentation_tools_pt.model import get_model
from segmentation_tools_pt.dataset import build_datasets   # see docs/agents/repo-guide.md for the arguments of your setup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = get_model(...)                       # same arguments as training
model.load_state_dict(torch.load('PATH/TO/best.pt', map_location=device, weights_only=True)['model_state'])
model.to(device)
val_ds = ...                                 # your validation dataset (see build_datasets)

def raw(sid):                                # keep 16-bit values: IMREAD_UNCHANGED, not the dataset's 8-bit read
    return cv2.imread(f'PATH/TO/raw/{sid}.png', cv2.IMREAD_UNCHANGED)

samples = samples_from_dataset(model, val_ds, device, class_names=['foreground'], raw_image_fn=raw)
print(build_report('T-real', 'reports/T-real', samples, seed=0))

AttributeError: 'ellipsis' object has no attribute 'startswith'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()